# Achtung
Das Notebook diente nur zum Aufsetzen der finalen Pipeline, um neue Etappen vorherzusagen und muss somit nicht nochmal ausgeführt werden.

Ziel war es hier eine Grundlage für eine Fahrerdatenbank zu schaffen.

In [1]:
import os
import time
import pickle
from datetime import datetime
import numpy as np
import pandas as pd
import sqlite3

In [4]:
PATH = "../../data/processed/26_cleaned_master_data.pkl"

df = pd.read_pickle(PATH)

# Pickle in dem noch die Geburtsdaten drin sind
# wichtig, um age at race zu berechnen
df_birthdate = pd.read_pickle("../../data/processed/18_cleaned_master_data.pkl")


In [4]:
df.head(5)

,meta_race,meta_year,meta_url,rank,meta_rider_url,height,meta_name,meta_nationality,weight,meta_url_name,...,weather_temp_trend,weather_rain_prob_mean,weather_precipitation_mean,weather_humidity_mean,won_how_cat,gradient_final_km,lag_rider_points_season,lag_rider_rank_season,lag_race_competitiveness_median,lag_team_power_index
0,tour-de-france,2005,https://www.procyclingstats.com/race/tour-de-f...,1.0,tom-boonen,1.92,Tom Boonen,BE,82,tom-boonen,...,4.5,0.025,0.025,59.5,sprint_large_group,0.5,1593.0,11.0,216.0,374.0
1,tour-de-france,2005,https://www.procyclingstats.com/race/tour-de-f...,2.0,thor-hushovd,1.83,Thor Hushovd,NO,83,thor-hushovd,...,4.5,0.025,0.025,59.5,sprint_large_group,0.5,1224.0,17.0,216.0,201.0
2,tour-de-france,2005,https://www.procyclingstats.com/race/tour-de-f...,3.0,robbie-mcewen,1.71,Robbie McEwen,AU,67,robbie-mcewen,...,4.5,0.025,0.025,59.5,sprint_large_group,0.5,1678.0,8.0,216.0,139.0
3,tour-de-france,2005,https://www.procyclingstats.com/race/tour-de-f...,4.0,stuart-o-grady,1.76,Stuart O'Grady,AU,73,stuart-o-grady,...,4.5,0.025,0.025,59.5,sprint_large_group,0.5,1866.0,5.0,216.0,203.0
4,tour-de-france,2005,https://www.procyclingstats.com/race/tour-de-f...,5.0,luciano-andre-pagliarini-mendonca,1.74,Luciano André Pagliarini,BR,68,luciano-andre-pagliarini-mendonca,...,4.5,0.025,0.025,59.5,sprint_large_group,0.5,179.0,335.0,216.0,303.0


In [5]:
columns_list = df.columns.tolist()
print(columns_list)

['meta_race', 'meta_year', 'meta_url', 'rank', 'meta_rider_url', 'height', 'meta_name', 'meta_nationality', 'weight', 'meta_url_name', 'meta_departure', 'meta_arrival', 'distance', 'vertical_meters', 'one_day_races', 'gc', 'time_trial', 'sprint', 'climber', 'hills', 'stage_nr', 'meta_date', 'meta_departure_lat', 'meta_departure_lon', 'meta_arrival_lat', 'meta_arrival_lon', 'meta_rider_points_season', 'meta_rider_rank_season', 'meta_current_team', 'team_tier', 'age_at_race', 'rider_bmi', 'meta_race_competitiveness_median', 'meta_team_power_index', 'wind_stability_index', 'weather_temp_mean', 'weather_temp_trend', 'weather_rain_prob_mean', 'weather_precipitation_mean', 'weather_humidity_mean', 'won_how_cat', 'gradient_final_km', 'lag_rider_points_season', 'lag_rider_rank_season', 'lag_race_competitiveness_median', 'lag_team_power_index']


In [7]:
columns_list = df_birthdate.columns.tolist()
print(columns_list)

print(df_birthdate.tail(5))

['race', 'year', 'url', 'rank', 'rider_url', 'time_gap', 'birthdate', 'height', 'name', 'nationality', 'weight', 'url_name', 'departure', 'arrival', 'distance', 'vertical_meters', 'won_how', 'avg_speed', 'race_ranking', 'one_day_races', 'gc', 'time_trial', 'sprint', 'climber', 'hills', 'stage_nr', 'date', 'departure_lat', 'departure_lon', 'arrival_lat', 'arrival_lon', 'rider_points_season', 'rider_rank_season', 'current_team', 'departure_temp_mittel', 'departure_regen_mittel', 'departure_wind_mittel', 'departure_luftfeuchte_mittel', 'departure_niederschlag_mittel', 'departure_windrichtung_mittel', 'arrival_temp_mittel', 'arrival_regen_mittel', 'arrival_wind_mittel', 'arrival_luftfeuchte_mittel', 'arrival_niederschlag_mittel', 'arrival_windrichtung_mittel', 'time_gap_seconds', 'team_tier']
                   race  year  \
196043  vuelta-a-espana  2025   
196044  vuelta-a-espana  2025   
196045  vuelta-a-espana  2025   
196046  vuelta-a-espana  2025   
196047  vuelta-a-espana  2025   

 

# Extraktion der Median Werte für die Wetterdaten

In [6]:
print(df.columns.tolist())

['meta_race', 'meta_year', 'meta_url', 'rank', 'meta_rider_url', 'height', 'meta_name', 'meta_nationality', 'weight', 'meta_url_name', 'meta_departure', 'meta_arrival', 'distance', 'vertical_meters', 'one_day_races', 'gc', 'time_trial', 'sprint', 'climber', 'hills', 'stage_nr', 'meta_date', 'meta_departure_lat', 'meta_departure_lon', 'meta_arrival_lat', 'meta_arrival_lon', 'meta_rider_points_season', 'meta_rider_rank_season', 'meta_current_team', 'team_tier', 'age_at_race', 'rider_bmi', 'meta_race_competitiveness_median', 'meta_team_power_index', 'wind_stability_index', 'weather_temp_mean', 'weather_temp_trend', 'weather_rain_prob_mean', 'weather_precipitation_mean', 'weather_humidity_mean', 'won_how_cat', 'gradient_final_km', 'lag_rider_points_season', 'lag_rider_rank_season', 'lag_race_competitiveness_median', 'lag_team_power_index']


In [9]:
weather_cols = [
    "weather_temp_mean",
    "weather_humidity_mean",
    "weather_rain_prob_mean",
    "weather_precipitation_mean",
    "wind_stability_index",
    "weather_temp_trend"
]

# Sicherstellen, dass wir nur die 3 Grand Tours betrachten
grand_tours = ["giro-d-italia", "tour-de-france", "vuelta-a-espana"]
df_gt = df[df["meta_race"].str.lower().isin(grand_tours)].copy()

# Mediane gruppiert nach Rennen berechnen
df_weather_medians = df_gt.groupby("meta_race")[weather_cols].median()

print("\n==================================================================")
print(" HISTORISCHE WETTER-MEDIANE AUS DEINEM PKL-DATENRAHMEN")
print("==================================================================")
print(df_weather_medians.round(4).to_string())
print("==================================================================")


 HISTORISCHE WETTER-MEDIANE AUS DEINEM PKL-DATENRAHMEN
                 weather_temp_mean  weather_humidity_mean  weather_rain_prob_mean  weather_precipitation_mean  wind_stability_index  weather_temp_trend
meta_race                                                                                                                                              
giro-d-italia                18.33                  59.25                   0.040                       0.040                0.1990               -1.25
tour-de-france               22.95                  54.00                   0.015                       0.015                0.1469                0.12
vuelta-a-espana              24.24                  50.50                   0.000                       0.000                0.1736               -0.40


C:\Users\lukas\AppData\Local\Temp\ipykernel_27728\3604852986.py:15: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_weather_medians = df_gt.groupby("meta_race")[weather_cols].median()


## Extraktion der Fahrerspalten
- Fahrer ab 2022
- reicht um die Profile für zukünftige Etappen abzudecken

In [7]:
df_filtered = df[df['meta_year'] > 2022].copy()
# ausgewählte Liste der relevanten Fahrer-Features definieren
fahrer_features = [
    'meta_rider_url',          # Die eindeutige Fahrer-ID von PCS
    'meta_name',               # Klarname des Fahrers
    'meta_url_name',            # Kurzname/URL-Name
    'meta_year',
    'height',                  # Körpergröße
    'weight',                  # Gewicht
    'rider_bmi',               # Berechneter BMI aus dem Datensatz
    'meta_nationality',         # Nationalität
    'meta_current_team',       # Aktuelles Team
    'team_tier',               # Team-Kategorie (Tier)
    'lag_rider_points_season', # Saisonpunkte des Vorjahres
    'lag_rider_rank_season'    # Saisonrang des Vorjahres
]

# DataFrame auf diese Spalten reduzieren
df_riders_raw = df_filtered[fahrer_features].copy()

# Spaltennamen für eine saubere, einheitliche Struktur normalisieren
df_riders_raw.rename(columns={
    'meta_rider_url': 'rider_url',
    'meta_nationality': 'nationality',
    'meta_current_team': 'current_team'
}, inplace=True)



# Duplikate entfernen: Da die Lags und Teams sich über die Jahre (2005-2024) geändert haben,
# behalten wir pro Fahrer NUR den aktuellsten Zustand (den letzten Eintrag im DataFrame)
df_riders_clean = df_riders_raw.drop_duplicates(subset=['rider_url'], keep='last').reset_index(drop=True)

# Kontrolle der Daten anzeigen
print(f"Anzahl einzigartiger Fahrer-Profile extrahiert: {df_riders_clean.shape[0]}")
df_riders_clean.head(20)

Anzahl einzigartiger Fahrer-Profile extrahiert: 634


,rider_url,meta_name,meta_url_name,meta_year,height,weight,rider_bmi,nationality,current_team,team_tier,lag_rider_points_season,lag_rider_rank_season
0,jacopo-guarnieri,Jacopo Guarnieri,jacopo-guarnieri,2023,1.90,80,22.160665,IT,Lotto Dstny,continental,49.0,761.0
1,antonio-pedrero,Antonio Pedrero,antonio-pedrero,2023,1.77,60,19.151585,ES,Movistar Team,elite,301.0,203.0
2,peter-sagan,Peter Sagan,peter-sagan,2023,1.82,78,23.547881,SK,TotalEnergies,continental,443.0,132.0
3,sam-welsford,Sam Welsford,sam-welsford,2023,1.82,79,23.849777,AU,Team dsm - firmenich,elite,193.0,300.0
4,benoit-cosnefroy,Benoît Cosnefroy,benoit-cosnefroy,2023,1.79,65,20.286508,FR,AG2R Citroën Team,elite,1002.0,30.0
5,florian-vermeersch,Florian Vermeersch,florian-vermeersch,2023,1.93,85,22.819405,BE,Lotto Dstny,continental,251.0,246.0
6,daniel-oss,Daniel Oss,daniel-oss,2023,1.91,75,20.558647,IT,TotalEnergies,continental,74.0,597.0
7,edvald-boasson-hagen,Edvald Boasson Hagen,edvald-boasson-hagen,2023,1.80,75,23.148148,NO,TotalEnergies,continental,310.0,193.0
8,gorka-izagirre,Gorka Izagirre,gorka-izagirre,2023,1.76,66,21.306818,ES,Movistar Team,elite,85.0,541.0
9,anthony-delaplace,Anthony Delaplace,anthony-delaplace,2023,1.81,65,19.840664,FR,Team Arkéa Samsic,elite,190.0,303.0


## Geburtstag zurück holen

In [8]:
df_birthdate.head(5)

,race,year,url,rank,rider_url,time_gap,birthdate,height,name,nationality,...,departure_niederschlag_mittel,departure_windrichtung_mittel,arrival_temp_mittel,arrival_regen_mittel,arrival_wind_mittel,arrival_luftfeuchte_mittel,arrival_niederschlag_mittel,arrival_windrichtung_mittel,time_gap_seconds,team_tier
0,tour-de-france,2005,https://www.procyclingstats.com/race/tour-de-f...,1.0,tom-boonen,3:51:313:51:31,1980-10-15,1.92,Tom Boonen,BE,...,0.0,328.25,28.32,0.05,6.22,50.0,0.05,277.25,0,elite
1,tour-de-france,2005,https://www.procyclingstats.com/race/tour-de-f...,2.0,thor-hushovd,",,0:00",1978-01-18,1.83,Thor Hushovd,NO,...,0.0,328.25,28.32,0.05,6.22,50.0,0.05,277.25,0,elite
2,tour-de-france,2005,https://www.procyclingstats.com/race/tour-de-f...,3.0,robbie-mcewen,",,0:00",1972-06-24,1.71,Robbie McEwen,AU,...,0.0,328.25,28.32,0.05,6.22,50.0,0.05,277.25,0,elite
3,tour-de-france,2005,https://www.procyclingstats.com/race/tour-de-f...,4.0,stuart-o-grady,",,0:00",1973-08-06,1.76,Stuart O'Grady,AU,...,0.0,328.25,28.32,0.05,6.22,50.0,0.05,277.25,0,elite
4,tour-de-france,2005,https://www.procyclingstats.com/race/tour-de-f...,5.0,luciano-andre-pagliarini-mendonca,",,0:00",1978-04-18,1.74,Luciano André Pagliarini,BR,...,0.0,328.25,28.32,0.05,6.22,50.0,0.05,277.25,0,elite


In [9]:
# relevanten Spalten extrahieren
df_birthdate_clean = df_birthdate[['rider_url', 'birthdate']].copy()

# Duplikate in den Geburtsdaten entfernen, damit jeder Fahrer nur einmal existiert
df_birthdate_clean = df_birthdate_clean.drop_duplicates(subset=['rider_url'])

# Direkt zusammenfügen (Left Join) ohne vorherige Textbereinigung
df_riders_final = pd.merge(
    df_riders_clean,
    df_birthdate_clean,
    left_on='rider_url',  # Spaltenname aus deiner Haupttabelle
    right_on='rider_url',      # Spaltenname aus der Geburtsdaten-Tabelle
    how='left'
)



# Kontrolle anzeigen

print(" Merge erfolgreich")
print("==================================================================")
print(f"Anzahl Fahrer im finalen Datensatz: {df_riders_final.shape[0]}")
print(f"Fehlende Geburtsdaten (NaNs): {df_riders_final.isna().sum()}")
print("------------------------------------------------------------------")
df_riders_final.head(5)

 Merge erfolgreich
Anzahl Fahrer im finalen Datensatz: 634
Fehlende Geburtsdaten (NaNs): rider_url                  0
meta_name                  0
meta_url_name              0
meta_year                  0
height                     0
weight                     0
rider_bmi                  0
nationality                0
current_team               0
team_tier                  0
lag_rider_points_season    0
lag_rider_rank_season      0
birthdate                  0
dtype: int64
------------------------------------------------------------------


,rider_url,meta_name,meta_url_name,meta_year,height,weight,rider_bmi,nationality,current_team,team_tier,lag_rider_points_season,lag_rider_rank_season,birthdate
0,jacopo-guarnieri,Jacopo Guarnieri,jacopo-guarnieri,2023,1.90,80,22.160665,IT,Lotto Dstny,continental,49.0,761.0,1987-08-14
1,antonio-pedrero,Antonio Pedrero,antonio-pedrero,2023,1.77,60,19.151585,ES,Movistar Team,elite,301.0,203.0,1991-10-23
2,peter-sagan,Peter Sagan,peter-sagan,2023,1.82,78,23.547881,SK,TotalEnergies,continental,443.0,132.0,1990-01-26
3,sam-welsford,Sam Welsford,sam-welsford,2023,1.82,79,23.849777,AU,Team dsm - firmenich,elite,193.0,300.0,1996-01-19
4,benoit-cosnefroy,Benoît Cosnefroy,benoit-cosnefroy,2023,1.79,65,20.286508,FR,AG2R Citroën Team,elite,1002.0,30.0,1995-10-17


Die Modell haben folgende Features, die die Fahrer betreffen:

- 'team_tier'
- 'age_at_race'
- 'rider_bmi'
- 'lag_rider_points_season'
- 'lag_rider_rank_season'

In [10]:
# Datei speichern

backup_folder = "../../data/processed"
backup_path = os.path.join(backup_folder, "df_riders_final_production.pkl")
df_riders_final.to_pickle(backup_path)

## SQLITE DB erstellen

In [11]:
db_folder = "../../data/databases"
os.makedirs(db_folder, exist_ok=True)
db_path = os.path.join(db_folder, "cycling_production.db")

# Verbindung aufbauen
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

try:
    # Sauberer Neustart der Tabellen
    cursor.execute("DROP TABLE IF EXISTS r_master")
    cursor.execute("DROP TABLE IF EXISTS lags_historical")

    # Tabelle 1: Stammdaten
    cursor.execute('''
        CREATE TABLE r_master (
            rider_url TEXT PRIMARY KEY,
            meta_name TEXT,
            meta_url_name TEXT,
            height REAL,
            weight REAL,
            rider_bmi REAL,
            nationality TEXT,
            birthdate TEXT
        )
    ''')

    # Tabelle 2: Lags (team_tier ist TEXT)
    cursor.execute('''
        CREATE TABLE lags_historical (
            rider_url TEXT,
            season_year INTEGER,
            current_team TEXT,
            team_tier TEXT,
            lag_rider_points_season REAL,
            lag_rider_rank_season REAL,
            PRIMARY KEY (rider_url, season_year)
        )
    ''')
    conn.commit()
    print("➔ Tabellenstrukturen SQLite erfolgreich neu angelegt.")
    print("➔ Befülle SQL-Datenbank mit DataFrame...")

    # A. Stammdaten schreiben
    for _, row in df_riders_final.iterrows():
        birthdate_str = None
        if pd.notna(row['birthdate']):
            birthdate_str = row['birthdate'].strftime('%Y-%m-%d') if hasattr(row['birthdate'], 'strftime') else str(row['birthdate'])

        cursor.execute('''
            INSERT INTO r_master (rider_url, meta_name, meta_url_name, height, weight, rider_bmi, nationality, birthdate)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?)
            ON CONFLICT(rider_url) DO UPDATE SET
                meta_name=excluded.meta_name,
                meta_url_name=excluded.meta_url_name,
                height=excluded.height,
                weight=excluded.weight,
                rider_bmi=excluded.rider_bmi,
                nationality=excluded.nationality,
                birthdate=excluded.birthdate
        ''', (
            row['rider_url'], row['meta_name'], row['meta_url_name'],
            row['height'], row['weight'], row['rider_bmi'],
            row['nationality'], birthdate_str
        ))

    # B. Lags schreiben
    for _, row in df_riders_final.iterrows():
        year = int(row['meta_year']) if pd.notna(row['meta_year']) else 2025
        points = row['lag_rider_points_season'] if pd.notna(row['lag_rider_points_season']) else 0.0
        rank_pos = row['lag_rider_rank_season'] if pd.notna(row['lag_rider_rank_season']) else 999.0
        tier_str = str(row['team_tier']) if pd.notna(row['team_tier']) else "unknown"

        cursor.execute('''
            INSERT INTO lags_historical (rider_url, season_year, current_team, team_tier, lag_rider_points_season, lag_rider_rank_season)
            VALUES (?, ?, ?, ?, ?, ?)
            ON CONFLICT(rider_url, season_year) DO UPDATE SET
                current_team=excluded.current_team,
                team_tier=excluded.team_tier,
                lag_rider_points_season=excluded.lag_rider_points_season,
                lag_rider_rank_season=excluded.lag_rider_rank_season
        ''', (
            row['rider_url'], year, row['current_team'], tier_str, points, rank_pos
        ))

    # Wenn alles geklappt hat, Änderungen einfrieren
    conn.commit()
    print(f"\n==================================================================\n"
          f"ERFOLG: {len(df_riders_final)} Fahrer-Profile erfolgreich integriert!\n"
          f"==================================================================")

except Exception as e:
    print(f"Fehler während der Transaktion: {e}")
    print("➔ Führe Rollback aus...")
    conn.rollback() # Macht unvollständige Änderungen rückgängig

finally:
    # EGAL WAS PASSIERT: Die Verbindung wird am Ende IMMER geschlossen!
    conn.close()
    print("Datenbankverbindung sicher geschlossen.")

➔ Tabellenstrukturen SQLite erfolgreich neu angelegt.
➔ Befülle SQL-Datenbank mit DataFrame...

ERFOLG: 634 Fahrer-Profile erfolgreich integriert!
Datenbankverbindung sicher geschlossen.
